# Resume training on Colab A100

**Pre-requisites** (set once before running this notebook):
1. Run `prep_data_hub.py` locally to upload `train/val/test.parquet` to a HF dataset repo.
2. From the most recent training pod, push the latest checkpoint dir to a HF model repo (e.g. `junho5400/svg-finetune-qwen-7b-lora-resume`) using `hf upload <repo> outputs/checkpoint/checkpoint-XXXX --include '*'`.

**This notebook**: Runtime → A100, then Run all. Each cell sets up env, downloads data + checkpoint, runs training, pushes the new checkpoint at end. Designed to fit in one ~10-12 hour Colab Pro session.

**For multi-session training**: the LATEST cell pushes the new checkpoint to `junho5400/svg-finetune-qwen-7b-lora-resume` (overwriting the prior one). Next session, just open this notebook again — it'll pull the new checkpoint and continue.

In [ ]:
# === Config — edit these to match your repos ===
GIT_REPO = 'https://github.com/junho5400/svg_finetune.git'
DATA_REPO = 'junho5400/svg-finetune-data'             # HF dataset repo
RESUME_REPO = 'junho5400/svg-finetune-qwen-7b-lora-resume'  # HF model repo with latest checkpoint
WANDB_ENTITY = 'junho5400-northwestern-university'
MAX_STEPS = 30000  # session budget — increase if you want longer; A100 typically does ~3000 steps/hour

In [ ]:
# === Auth (paste your tokens) ===
from getpass import getpass
import os
os.environ['HF_TOKEN'] = getpass('HF write token: ')
os.environ['WANDB_API_KEY'] = getpass('WandB API key: ')
os.environ['WANDB_ENTITY'] = WANDB_ENTITY

In [ ]:
# === Clone repo + install pinned deps ===
!cd /content && git clone {GIT_REPO} svg_finetune || (cd svg_finetune && git pull)
!cd /content/svg_finetune && pip install -q -r requirements.txt

In [ ]:
# === Pull data parquets from HF dataset repo ===
from huggingface_hub import hf_hub_download
import os

DATA_DIR = '/content/svg_finetune/data'
os.makedirs(DATA_DIR, exist_ok=True)
for fname in ['train.parquet', 'val.parquet', 'test.parquet']:
    print(f'fetching {fname}...')
    hf_hub_download(repo_id=DATA_REPO, filename=fname,
                    repo_type='dataset', local_dir=DATA_DIR,
                    token=os.environ.get('HF_TOKEN'))
print('data ready')

In [ ]:
# === Pull latest checkpoint from HF Hub ===
# Snapshot the resume repo to a temp dir, read trainer_state.json to get the
# step number, then rename to "checkpoint-{step}". HF Trainer's auto-resume
# logic only matches dirs named `checkpoint-{number}` — anything else (like
# "checkpoint-resume") is silently ignored, causing fresh training.
import os, json, shutil
from huggingface_hub import snapshot_download

OUT_CKPT_DIR = '/content/svg_finetune/outputs/checkpoint'
TMP_DIR = f'{OUT_CKPT_DIR}/_tmp_resume_download'
os.makedirs(OUT_CKPT_DIR, exist_ok=True)

print(f'downloading from {RESUME_REPO}...')
snapshot_download(repo_id=RESUME_REPO, local_dir=TMP_DIR,
                  token=os.environ.get('HF_TOKEN'))
print('files downloaded:')
for f in sorted(os.listdir(TMP_DIR)):
    print(f'  {f}')

state_path = f'{TMP_DIR}/trainer_state.json'
if os.path.exists(state_path):
    with open(state_path) as f:
        step = int(json.load(f).get('global_step', 0))
    print(f'\ncheckpoint is at step {step}')
else:
    # Without trainer_state.json, Trainer can load model weights but not the
    # step counter / optimizer / LR schedule. Resume becomes a warm start.
    print('\nWARNING: trainer_state.json missing — partial resume only')
    print('         (model weights load but step counter resets to 0)')
    step = 0

ckpt_dir = f'{OUT_CKPT_DIR}/checkpoint-{step}'
if os.path.exists(ckpt_dir):
    shutil.rmtree(ckpt_dir)
os.rename(TMP_DIR, ckpt_dir)
print(f'\nplaced at: {ckpt_dir}')

In [ ]:
# === Tune for A100: bigger batches + skip grad_checkpointing ===
# Detect GPU memory and set env vars that train.py reads as overrides.
#
# IMPORTANT: max_length stays at 1024 (matching prior RunPod training).
# Changing it mid-training would shift the data distribution: max_length is a
# training-time filter that decides which examples are included. The model has
# only seen <=1024-token glyphs so far; introducing longer ones now would cause
# a loss spike and unlearning. Use A100's extra VRAM for bigger batches instead.
import torch, os
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {torch.cuda.get_device_name(0)} ({vram_gb:.0f} GB)')

if vram_gb > 70:
    # A100 80GB or H100 80GB — large batch, no grad-ckpt
    os.environ['SVG_BATCH_SIZE'] = '8'
    os.environ['SVG_GRAD_ACCUM'] = '2'      # effective batch 16, same as before
    os.environ['SVG_GRAD_CKPT'] = 'false'   # plenty of memory; skip the 30% slowdown
elif vram_gb > 35:
    # A100 40GB — moderate batch, no grad-ckpt
    os.environ['SVG_BATCH_SIZE'] = '4'
    os.environ['SVG_GRAD_ACCUM'] = '4'
    os.environ['SVG_GRAD_CKPT'] = 'false'
else:
    # T4 or smaller — keep 4090-style conservative settings
    os.environ['SVG_BATCH_SIZE'] = '2'
    os.environ['SVG_GRAD_ACCUM'] = '8'
    os.environ['SVG_GRAD_CKPT'] = 'true'

# Lock max_length to match RunPod training — DON'T change this mid-run
os.environ['SVG_MAX_LENGTH'] = '1024'

# === Resilience: push full checkpoints to HF Hub on every local save ===
# - hub_strategy=every_save: latest checkpoint always on Hub. If Colab disconnects,
#   lose at most save_steps worth of progress (not the whole session).
# - SAVE_ONLY_MODEL=false: include trainer_state.json + optimizer.pt + scheduler.pt.
#   Without these, next session can only warm-start (loses LR schedule + Adam
#   momentum + step counter). The ~940 MB checkpoint vs ~309 MB adapter-only is
#   worth it for full resume capability across sessions.
# - save_steps=2000: ~40 min on A100. Larger interval reduces upload bandwidth
#   (~1.4 GB/hour vs ~2.8 GB/hour at save_steps=1000) while still capping disconnect
#   loss to <1 hour of progress.
os.environ['SVG_HUB_STRATEGY'] = 'every_save'
os.environ['SVG_HUB_REPO'] = RESUME_REPO       # overwrite the resume repo each save
os.environ['SVG_SAVE_ONLY_MODEL'] = 'false'    # need full state for cross-session resume
os.environ['SVG_SAVE_STEPS'] = '2000'

# Session step budget
os.environ['SVG_MAX_STEPS_OVERRIDE'] = str(MAX_STEPS)

print(f'\nSettings for this session:')
for k in ['SVG_BATCH_SIZE', 'SVG_GRAD_ACCUM', 'SVG_MAX_LENGTH', 'SVG_GRAD_CKPT',
          'SVG_HUB_STRATEGY', 'SVG_HUB_REPO', 'SVG_SAVE_ONLY_MODEL',
          'SVG_SAVE_STEPS', 'SVG_MAX_STEPS_OVERRIDE']:
    print(f'  {k} = {os.environ.get(k)}')

In [ ]:
# === Run training ===
# Auto-resume kicks in because cell-5 placed a checkpoint-{step} dir under
# outputs/checkpoint/. train.py calls trainer.train(resume_from_checkpoint=True)
# which makes HF Trainer auto-detect that dir and continue from its step.
!cd /content/svg_finetune && python train.py 2>&1 | tee train.log

In [ ]:
# === Push new latest checkpoint to HF Hub for next session's resume ===
# Note: with hub_strategy=every_save (set in cell-6), checkpoints are pushed
# automatically during training. This cell is a manual safety push at end of
# session — useful if the run was stopped between save_steps boundaries.
import glob, json
checkpoints = sorted(
    glob.glob(f'{OUT_CKPT_DIR}/checkpoint-*'),
    key=lambda p: int(p.rsplit('-', 1)[-1])  # sort by step number
)
if not checkpoints:
    print('no checkpoint dir found — run probably failed early. Check train.log above.')
else:
    latest = checkpoints[-1]
    print(f'latest checkpoint: {latest}')
    print(f'pushing to {RESUME_REPO}...')
    !cd {latest} && hf upload {RESUME_REPO} . --include '*'

In [ ]:
# === Inspect tail of training log (last 30 lines) ===
!tail -30 /content/svg_finetune/train.log